In [2]:
import argparse
import os
import sys
import subprocess
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import hypergeom
warnings.filterwarnings("ignore")

### 处理df

In [3]:
df = pd.read_csv('../DB/shoot.atlas/cell-type-foundation-genes.csv')
df.head()

,EC,VC_Ph,VC_Xy,VC_PC,MC,VC_CC,GC,PC_S,PC_G2
0,AT5G45950,AT2G37590,AT5G19530,AT5G09220,AT2G05070,AT2G35635,AT2G04570,AT1G01370,AT1G02690
1,AT1G29660,AT2G28510,AT1G68810,AT5G63850,AT3G27690,AT1G31340,AT1G75900,AT1G04020,AT1G02730
2,AT3G16570,AT3G14570,AT3G25710,AT5G49630,AT5G54270,AT5G40370,AT1G56580,AT1G07790,AT1G07880
3,AT4G28780,AT3G59100,AT4G32880,AT1G20823,AT1G29910,AT5G62680,AT2G26250,AT1G14900,AT1G08560
4,AT1G75900,AT5G62940,AT1G52150,AT3G56240,AT2G34430,AT1G69850,AT1G66400,AT1G47210,AT1G18370


In [7]:
# 直接 melt 所有细胞列，将列名转为 'Cell'，单元格内的基因转为 'Gene'
df_result = df.melt(var_name='Cell', value_name='Gene')

# 调换一下列的顺序，变成你要求的：第一列基因，第二列细胞名
df_result = df_result[['Gene', 'Cell']]
# 1. 去掉 'Gene' 列中包含 NaN 的所有行
df_result = df_result.dropna(subset=['Gene'])
# 2. 如果你想顺便把索引重置成连续的 0, 1, 2...
df_result = df_result.reset_index(drop=True)
# 查看处理后的前几行
df_result

,Gene,Cell
0,AT5G45950,EC
1,AT1G29660,EC
2,AT3G16570,EC
3,AT4G28780,EC
4,AT1G75900,EC
...,...,...
590,AT5G45700,PC_G2
591,AT5G48310,PC_G2
592,AT5G51600,PC_G2
593,AT5G60930,PC_G2


In [9]:
df_result.to_csv('../DB/shoot.atlas/cell-type-foundation-genes2.csv', index=False)

### 测试xspeciesspaner

In [1]:
# ============================================================
# Step 1: 构建评分基因集
# ============================================================
def build_scoring_gene_sets(foundation_csv, saturn_genes=None):
    """
    读取 foundation genes CSV，构建评分基因集字典。

    CSV 格式: 两列 [gene, cell_type]
    每行表示一个基因属于某个细胞类型的 foundation gene set。

    可选: 与 SATURN 识别的基因取交集过滤。
    """
    print("\n" + "=" * 60)
    print("Step 1: 构建评分基因集 (Scoring Gene Set Construction)")
    print("=" * 60)

    df = pd.read_csv(foundation_csv)
    required_cols = {"gene", "cell_type"}
    if not required_cols.issubset(df.columns):
        print(f"[错误] foundation_csv 必须包含列: {required_cols}")
        print(f"  当前列: {list(df.columns)}")
        sys.exit(1)

    # 构建 {cell_type: set(genes)} 字典
    gene_sets = {}
    for cell_type, group in df.groupby("cell_type"):
        genes = set(group["gene"].dropna().str.strip())
        gene_sets[cell_type] = genes

    print(f"  读取 foundation genes: {len(df)} 行, {len(gene_sets)} 个细胞类型")
    for ct, genes in gene_sets.items():
        print(f"    - {ct}: {len(genes)} 个基因")

    # 可选: 与 SATURN 识别的基因取交集
    if saturn_genes:
        saturn_set = set()
        with open(saturn_genes) as f:
            for line in f:
                gene = line.strip()
                if gene:
                    saturn_set.add(gene)
        print(f"\n  SATURN 基因列表: {len(saturn_set)} 个基因")

        filtered_sets = {}
        for ct, genes in gene_sets.items():
            overlap = genes & saturn_set
            filtered_sets[ct] = overlap
            print(f"    - {ct}: {len(genes)} -> {len(overlap)} (交集后)")
        gene_sets = filtered_sets

    # 统计所有 foundation genes 的并集
    all_foundation_genes = set()
    for genes in gene_sets.values():
        all_foundation_genes.update(genes)

    print(f"\n  所有 foundation genes 并集: {len(all_foundation_genes)} 个基因")
    print(f"  Step 1 完成!")
    return gene_sets, all_foundation_genes